In [2]:
import warnings

from tqdm import tqdm

warnings.filterwarnings("ignore")
import os
import shutil

import matplotlib
import pandas as pd

import codecs  # this is used for file operations
import gc
import multiprocessing
import pickle
import pickle as pkl
import random as r
import re
from datetime import datetime as dt
from multiprocessing import Process  # this is used for multithreading

import dask.dataframe as dd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import scipy.sparse
import seaborn as sns
from itertools import product
from pathlib import Path
from nltk.util import ngrams
from sklearn import preprocessing
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2, f_regression
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, log_loss
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder

In [6]:
%%time

opcodes_for_bigram = ['jmp', 'mov', 'retf', 'push', 'pop', 'xor', 'retn', 'nop', 'sub', 'inc', 'dec', 'add','imul', 'xchg', 'or', 'shr', 'cmp', 'call', 'shl', 'ror', 'rol', 'jnb','jz','rtn','lea','movzx']

# Converting list to dictionary for faster runtime
dict_asm_opcodes = dict(zip(opcodes_for_bigram, [1 for i in range(len(opcodes_for_bigram))]))

if not os.path.isdir("opcodes_asm_files"):
    os.mkdir('opcodes_asm_files')

'''

Noting first that the asm files contains :

1. Address
2. Segments
3. Opcodes
4. Registers
5. function calls
6. APIs

Calculating opcode sequences for each asm file and save in form of a text file, so that we can process the ASM files as text files

In that text file, each row corresponds to respective file. 

Noting, in asm files the opcodes_for_bigram were not placed side by side, instead there are few words between two opcodes. i.e. The Opcodes occurs with an interval.

So during extraction of opcodes_for_bigram we need to preserve the sequence information. 

e.g. which opcode prcede another opcode or which opcode is followed is followed by another opcode.

Based on this, a bigram data-matrix of vectors is to be derived containing the bigram sequence info on each file.

'''

def calculate_bigram(bigram_tokens):
    sentence=""
    vocabulary_list_for_byte_bigrams=[]
    for i in tqdm(range(len(bigram_tokens))):
        for j in range(len(bigram_tokens)):
            bigram=bigram_tokens[i]+" "+bigram_tokens[j]
            sentence=sentence+bigram+","
            vocabulary_list_for_byte_bigrams.append(bigram)
    return vocabulary_list_for_byte_bigrams

def calculate_sequence_of_opcodes():
    # dict_asm_opcodes = dict(zip(OPCODES, [1 for i in range(len(OPCODES))]))
    raw_data_path = Path("..") / "data" / "raw_data"
    subfolders = ["first", "second", "third", "fourth", "fifth"]
    target_dir = Path("opcodes_asm_files")
    target_dir.mkdir(parents=True, exist_ok=True)

    for folder in tqdm(subfolders, desc="Totla Folder"):
        folder_path = raw_data_path / folder

        asm_files = [f for f in os.listdir(folder_path) if f.endswith(".asm")]
    
    # asm_file_names=os.listdir('asmFiles')
        for this_asm_file in tqdm(asm_files):
            each_asm_opcode_file = open("opcodes_asm_files/{}_opcode_asm_bi_grams.txt".format(this_asm_file.split('.')[0]), "w+")
            sequence_of_opcodes = ""
            with codecs.open(folder_path / this_asm_file, "r", encoding='cp1252', errors ='replace') as asm_file:
                for lines in asm_file:
                    
                    line = lines.rstrip().split()            
                    
                    for word in line:
                        if dict_asm_opcodes.get(word)==1:
                            sequence_of_opcodes += word + ' '
            each_asm_opcode_file.write(sequence_of_opcodes + "\n")
            each_asm_opcode_file.close()
    
calculate_sequence_of_opcodes()

opcodes_asm__bigram_vocabulary = calculate_bigram(opcodes_for_bigram)

100%|███████████████████████████████████████████████████████████████████████████████| 26/26 [00:00<00:00, 13000.94it/s]

CPU times: total: 3h 51min 1s
Wall time: 4h 50s


In [10]:
vectorizer_opcode = CountVectorizer(
    tokenizer=lambda x: x.split(),
    lowercase=False,
    ngram_range=(2, 2),
    vocabulary=opcodes_asm__bigram_vocabulary,
)  # Noting, without "tokenizer=lambda x: x.split()", "??" would not get vectorized correctly

file_list_opcode = os.listdir("opcodes_asm_files")
feature_names_array = vectorizer_opcode.get_feature_names_out()
opcode_features = ["ID"] + list(feature_names_array)

opcodes_asm_bigram_df = pd.DataFrame(columns=opcode_features)

with open(
    "featurization/opcodes_asm_bigram_df.csv", mode="w"
) as opcodes_asm_bigram_df:

    opcodes_asm_bigram_df.write(",".join(map(str, opcode_features)))

    opcodes_asm_bigram_df.write("\n")

    for _, this_asm_file in tqdm(enumerate(file_list_opcode)):

        this_file_id = this_asm_file.split("_")[0]  # ID of each this_asm_file

        this_asm_file = open("opcodes_asm_files/" + this_asm_file)

        corpus_opcodes_from_this_asm_file = [
            this_asm_file.read().replace("\n", " ").lower()
        ]  # Variable to hold all opcodes for a given this_asm_file

        bigrams_opcodes_asm = vectorizer_opcode.transform(
            corpus_opcodes_from_this_asm_file
        )  # Returning a sparse vector holding all bigram counts from corpus_opcodes_from_this_asm_file

        # Update each row of the dataframe with the bigram counts of the respective this_asm_file
        # And return a dense ndarray representation of this matrix. Because,
        # CountVectorizer produces a sparse representation of the counts using scipy.sparse.csr_matrix
        row = scipy.sparse.csr_matrix(bigrams_opcodes_asm).toarray()

        opcodes_asm_bigram_df.write(
            ",".join(map(str, [this_file_id] + list(row[0])))
        )  # Write a single row in the CSV this_asm_file

        opcodes_asm_bigram_df.write("\n")

        this_asm_file.close()

10868it [01:10, 154.44it/s]


,ID,jmp jmp,jmp mov,jmp retf,jmp push,jmp pop,jmp xor,jmp retn,jmp nop,jmp sub,...,movzx cmp,movzx call,movzx shl,movzx ror,movzx rol,movzx jnb,movzx jz,movzx rtn,movzx lea,movzx movzx
0,01azqd4InC7m9JpocGv5,440,192,0,6,0,17,0,0,24,...,0,0,0,0,0,0,0,0,0,0
1,01IsoiSMh5gxyDYTl4CB,0,32,0,3,1,3,1,0,0,...,1,0,0,0,0,0,0,0,0,0
2,01jsnpXSAlgw6aPeDxrU,0,0,0,0,0,0,0,0,0,...,12,0,0,0,0,0,3,0,17,12
3,01kcPWA9K2BOxQeS5Rju,0,5,0,1,0,2,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,01SuzwMJEIXsK7A8dQbl,5,57,1,4,1,1,0,0,0,...,11,0,0,0,0,0,0,0,0,4


In [3]:
opcodes_asm_bigram_df = pd.read_csv(
    "featurization/opcodes_asm_bigram_df.csv"
)

opcodes_asm_bigram_df.head()

,ID,jmp jmp,jmp mov,jmp retf,jmp push,jmp pop,jmp xor,jmp retn,jmp nop,jmp sub,...,movzx cmp,movzx call,movzx shl,movzx ror,movzx rol,movzx jnb,movzx jz,movzx rtn,movzx lea,movzx movzx
0,01azqd4InC7m9JpocGv5,440,192,0,6,0,17,0,0,24,...,0,0,0,0,0,0,0,0,0,0
1,01IsoiSMh5gxyDYTl4CB,0,32,0,3,1,3,1,0,0,...,1,0,0,0,0,0,0,0,0,0
2,01jsnpXSAlgw6aPeDxrU,0,0,0,0,0,0,0,0,0,...,12,0,0,0,0,0,3,0,17,12
3,01kcPWA9K2BOxQeS5Rju,0,5,0,1,0,2,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,01SuzwMJEIXsK7A8dQbl,5,57,1,4,1,1,0,0,0,...,11,0,0,0,0,0,0,0,0,4


In [5]:
# 1. 讀取標籤並合併 (這時候 final_df 包含所有資料)
with open('featurization/class_labels.pkl', 'rb') as file:
    class_labels = pkl.load(file)
y_labels = class_labels.rename(columns={"Id": "ID", "Class": "Class"})
full_merged_df = pd.merge(opcodes_asm_bigram_df, y_labels, on="ID", how="inner")

# 2. 讀取 ID 清單
train_ids_list = pd.read_csv("featurization/train_ids")["ID"]
test_ids_list = pd.read_csv("featurization/test_ids")["ID"]

# 3. 分別切出訓練集與測試集 (從 full_merged_df 切分)
train_df = full_merged_df[full_merged_df['ID'].isin(train_ids_list)].sort_values('ID')
test_df = full_merged_df[full_merged_df['ID'].isin(test_ids_list)].sort_values('ID')

# 4. 準備特徵 (X) 與標籤 (y)
X_train = train_df.drop(["ID", "Class"], axis=1)
y_train = train_df["Class"]
X_test = test_df.drop(["ID", "Class"], axis=1)
y_test = test_df["Class"]

# 5. 特徵選擇：Fit (只針對訓練集)
select_kbest_object = SelectKBest(score_func=chi2, k=500)
select_kbest_object.fit(X_train, y_train)

# 6. 特徵選擇：Transform (套用到兩邊，這步最關鍵！)
# 這會把特徵數從幾萬個縮減到 2000 個
X_train_reduced = select_kbest_object.transform(X_train)
X_test_reduced = select_kbest_object.transform(X_test)

# 7. (選做) 建立分數表，方便觀察
most_imp_byte_bigram_feature_score_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Score': select_kbest_object.scores_
}).sort_values(by='Score', ascending=False)

# 被選中的column name
selected_mask = select_kbest_object.get_support()
selected_feature_names = X_train.columns[selected_mask]

# 完整的train df 包含ID
X_train_reduced_df = pd.DataFrame(
    X_train_reduced, 
    columns=selected_feature_names, 
    index=X_train.index
)
X_train_reduced_df.insert(0, 'ID', train_df['ID'])

# 完整的test df 包含ID
X_test_reduced_df = pd.DataFrame(
    X_test_reduced, 
    columns=selected_feature_names, 
    index=X_test.index
)
X_test_reduced_df.insert(0, 'ID', test_df['ID'])

X_train_reduced_df.to_csv("featurization/featurization_final/top_500_asm_opcodes_bigram_df_train.csv",index=False)
X_test_reduced_df.to_csv("featurization/featurization_final/top_500_asm_opcodes_bigram_df_test.csv",index=False)

In [25]:
with open('featurization/class_labels.pkl', 'rb') as file:
    class_labels=pkl.load(file)

X_opcode_asm_bigram = opcodes_asm_bigram_df
y = class_labels

y

,Id,Class
0,01kcPWA9K2BOxQeS5Rju,1
1,04EjIdbPV5e1XroFOpiN,1
2,05EeG39MTRrI6VY21DPd,1
3,05rJTUWYAKNegBk2wE8X,1
4,0AnoOZDNbPXIr2MRBSCJ,1
...,...,...
10863,KFrZ0Lop1WDGwUtkusCi,9
10864,kg24YRJTB8DNdKMXpwOH,9
10865,kG29BLiFYPgWtpb350sO,9
10866,kGITL4OJxYMWEQ1bKBiP,9


In [27]:
with open('featurization/class_labels.pkl', 'rb') as file:
    class_labels=pkl.load(file)

X_opcode_asm_bigram = opcodes_asm_bigram_df
y = class_labels
y = y.rename(columns={"Id": "ID", "Class": "Class"})
final_df = pd.merge(X_opcode_asm_bigram, y, on="ID", how="inner")
# X_opcode_asm_bigram.head()

#Get the best 500 features using SelectKBest. 


# opcodes_asm_bigram_df = pd.read_csv("featurization/opcodes_asm_bigram_df.csv")
# X_opcode_asm_bigram = opcodes_asm_bigram_df
# base_path = Path("..") / "data" / "raw_data" / "trainLabels.csv"
# y = pd.read_csv(base_path)


X = final_df.drop(["ID", "Class"], axis=1)
y = final_df["Class"]

# le = LabelEncoder()
# y_encoded = le.fit_transform(y_raw.astype(str))





kbest_object = SelectKBest(score_func=chi2, k=500)

top_features=kbest_object.fit(X_opcode_asm_bigram.drop("ID", axis=1), y)

# Save a dataframe with the feature scores along with the feature names.
# And we will get the best fetures from this dataframe use to 
top_features_scores=pd.DataFrame(top_features.scores_)

# Now to get the original features names i.e. the names of all the columns we will need
# `X_opcode_asm_bigram.columns`
X_opcode_columns=pd.DataFrame(X_opcode_asm_bigram.columns)

# Now concat all  original features names as a column with another column
# which is "top_features_scores"
top_asm_opcode_bigram_df=pd.concat([X_opcode_columns, top_features_scores],axis=1)

# Give 2 Names for these 2 columns of data for this newly creaetd dataframe
top_asm_opcode_bigram_df.columns=["ASM_Opcode_Bigram_Top_Feature_Name","ASM_Opcode_Bigram_Top_Feature_Score"]

# Extract the largest 500 from this dataframw based on the values of "top_features_scores"
top_asm_opcode_bigram_df=top_asm_opcode_bigram_df.nlargest(500,"ASM_Opcode_Bigram_Top_Feature_Score")

top_asm_opcode_bigram_df.head()

,ASM_Opcode_Bigram_Top_Feature_Name,ASM_Opcode_Bigram_Top_Feature_Score
27,mov jmp,2.268131e+07
81,push retf,1.432259e+07
79,push jmp,9.157797e+06
95,push cmp,8.970330e+06
313,imul jmp,8.319890e+06


In [29]:
top_500_asm_bigram_features=list(top_asm_opcode_bigram_df["ASM_Opcode_Bigram_Top_Feature_Name"])

top_500_asm_bigram_df=pd.concat([X_opcode_asm_bigram["ID"], X_opcode_asm_bigram[top_500_asm_bigram_features]], axis=1)

# The "ID" column was being duplicated, hence need to remove that, and also the possibility of any other duplicated column
top_500_asm_bigram_df = top_500_asm_bigram_df.loc[:,~top_500_asm_bigram_df.columns.duplicated()]

top_500_asm_bigram_df.to_csv("featurization/featurization_final/top_500_asm_opcodes_bigram_df.csv",index=False)

top_500_asm_bigram_df.head()

,ID,mov jmp,push retf,push jmp,push cmp,imul jmp,mov add,mov retf,call jmp,mov cmp,...,jnb ror,retn call,sub rol,jmp shl,call call,inc add,shr xor,call or,nop mov,rol mov
0,01azqd4InC7m9JpocGv5,165,0,0,2,2,219,0,58,285,...,0,0,0,2,51,1,0,3,0,0
1,01IsoiSMh5gxyDYTl4CB,34,0,0,3,0,114,0,1,74,...,0,4,0,0,2,2,0,0,4,0
2,01jsnpXSAlgw6aPeDxrU,0,0,0,0,0,0,0,0,0,...,0,0,0,0,3,0,0,12,0,0
3,01kcPWA9K2BOxQeS5Rju,3,0,0,0,0,2,0,2,6,...,0,0,0,0,0,0,0,0,0,0
4,01SuzwMJEIXsK7A8dQbl,51,2,1,0,0,389,0,0,144,...,0,0,0,0,67,0,0,0,0,0
